# Process Documents

In [6]:
import os

import pandas as pd
from docling.document_converter import DocumentConverter

from agent_k.config.logger import logger

# Files that are taking too long to process. Exclude for now and process later if needed.
files_to_exclude = [
    "02711c830d14cbb8e365574b7aa58d792209a84d07eabf7c7354c342c93e260991.pdf"
]

df_inferlink_gt = pd.read_csv("data/processed/ground_truth/inferlink_ground_truth.csv")
record_ids = df_inferlink_gt["cdr_record_id"].tolist()

# Search each record ID in the 43-101 directory "data/raw/all_sources/43-101"
pdf_paths = []
for record_id in record_ids:
    for file in os.listdir("data/raw/all_sources/43-101"):
        if file in files_to_exclude:
            logger.info(f"Skipping because it is in the exclude list: {file}")
            continue
        if record_id in file:
            pdf_paths.append(os.path.join("data/raw/all_sources/43-101", file))

converter = DocumentConverter()

for i, pdf_path in enumerate(pdf_paths):
    logger.info(f"{i + 1}/{len(pdf_paths)}: Processing {pdf_path}")

    # Check if the markdown file already exists
    record_id = pdf_path.split("/")[-1].split(".")[0]
    if os.path.exists(os.path.join("data/processed/43-101", f"{record_id}.md")):
        logger.info("Skipping because it already exists")
        continue

    result = converter.convert(pdf_path)
    # Save the markdown to a file in processed/43-101
    with open(os.path.join("data/processed/43-101", f"{record_id}.md"), "w") as f:
        f.write(result.document.export_to_markdown())

2025-04-23 15:20:22.711 | INFO     | __main__:<module>:19 - Skipping because it is in the exclude list: 02711c830d14cbb8e365574b7aa58d792209a84d07eabf7c7354c342c93e260991.pdf
2025-04-23 15:20:22.712 | INFO     | __main__:<module>:19 - Skipping because it is in the exclude list: 02711c830d14cbb8e365574b7aa58d792209a84d07eabf7c7354c342c93e260991.pdf
2025-04-23 15:20:22.713 | INFO     | __main__:<module>:19 - Skipping because it is in the exclude list: 02711c830d14cbb8e365574b7aa58d792209a84d07eabf7c7354c342c93e260991.pdf
2025-04-23 15:20:22.714 | INFO     | __main__:<module>:19 - Skipping because it is in the exclude list: 02711c830d14cbb8e365574b7aa58d792209a84d07eabf7c7354c342c93e260991.pdf
2025-04-23 15:20:22.715 | INFO     | __main__:<module>:19 - Skipping because it is in the exclude list: 02711c830d14cbb8e365574b7aa58d792209a84d07eabf7c7354c342c93e260991.pdf
2025-04-23 15:20:22.716 | INFO     | __main__:<module>:19 - Skipping because it is in the exclude list: 02711c830d14cbb8e3655

## Refine Headers
1. Removing false positive headers in markdown splitters (e.g. "## Notes:")
2. Correct header levels

In [2]:
import os
import re
from pathlib import Path


def extract_headers_from_markdown(markdown_text):
    """
    Extract all headers from a markdown text.

    Args:
        markdown_text (str): The markdown text to parse

    Returns:
        list: A list of tuples containing (header_level, header_text)
    """
    # Regular expression to match markdown headers (# Header)
    header_pattern = re.compile(r"^(#{1,6})\s+(.+?)(?:\s+#+)?$", re.MULTILINE)

    # Find all headers in the text
    headers = []
    for match in header_pattern.finditer(markdown_text):
        level = len(match.group(1))  # Number of # characters
        text = match.group(2).strip()
        headers.append((level, text))

    return headers


# Example usage
markdown_dir = Path("data/processed/43-101")
sample_files = list(markdown_dir.glob("*.md"))[
    4:7
]  # Get first 3 files for demonstration

for file_path in sample_files:
    print(f"\nHeaders in {file_path.name}:")

    with open(file_path, "r", encoding="utf-8") as f:
        markdown_content = f.read()

    headers = extract_headers_from_markdown(markdown_content)

    for level, text in headers:
        print(f"{'  ' * (level - 1)}{'#' * level} {text}")


Headers in 025616a24d90f78dc37e943df3495bce368e062297a2e33a6f519524496c09c443.md:
  ## NOTE
  ## REPORT NI 43-101
  ## TECHNICAL REPORT ON THE
  ## MINERAL RESOURCES AND RESERVES OF THE
  ## LOS SANTOS MINE PROJECT,
  ## SPAIN
  ## TABLE OF CONTENTS
  ## LIST OF TABLES
  ## LIST OF FIGURES
  ## APPENDICES
  ## A Geostatistical Plots
  ## B Glossary of Terms
  ## 1 SUMMARY
  ## 1.1 Introduction and Overview
  ## 1.2 Ownership
  ## 1.3 Geology and Mineralization
  ## 1.4 Database and Resource Estimation
  ## 1.5 Mine Planning
  ## 1.6 Mineral Processing
  ## 1.7 Mineral Resource and Reserve Estimates
  ## Notes
  ## 1.8 Conclusions
  ## 2 INTRODUCTION
  ## 2.1 Introduction
  ## 2.2 Terms of Reference
  ## 2.3 Sources of Information
  ## 2.4 Units and Currency
  ## 2.5 Disclaimer
  ## 3 RELIANCE ON OTHER EXPERTS
  ## 4 PROPERTY DESCRIPTION AND LOCATION
  ## 5 ACCESSIBILITY, CLIMATE, LOCAL RESOURCES, INFRASTRUCTURE, PHYSIOGRAPHY
  ## 6 HISTORY
  ## 7 GEOLOGICAL SETTING AND MINERALIZATION


In [3]:
def correct_header_levels(headers):
    """
    Correct header levels in markdown headers based on specified heuristics:
    1. By default keep header level as level 2
    2. Numbered headers get levels based on their numbering depth relative to the highest level:
       - If highest level is "1", then "1" -> level 2, "1.1" -> level 3, etc.
       - If highest level is "1.0", then "1.0" -> level 2, "1.0.1" -> level 3, etc.
    3. Unnumbered headers between numbered headers get level one below the enclosing numbered headers
    4. Replace "## Notes:", "## Notes", or "## Note" with "Notes:"

    Args:
        headers: List of tuples (header_level, header_text)

    Returns:
        List of tuples with corrected (header_level, header_text)
    """
    if not headers:
        return []

    import re

    # First pass: Determine the highest level numbered header pattern
    highest_level_parts = None
    numbered_headers_info = []

    for i, (_, text) in enumerate(headers):
        number_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", text)
        if number_match:
            number = number_match.group(1)
            parts = number.split(".")

            # Store information about this numbered header
            numbered_headers_info.append((i, number, parts))

            # Update highest_level_parts if this is a higher level
            if highest_level_parts is None or len(parts) < len(highest_level_parts):
                highest_level_parts = parts

    # If no numbered headers found, highest level is None
    base_level_length = len(highest_level_parts) if highest_level_parts else 0

    # Second pass: Process all headers with the determined base level
    corrected = []
    current_section_level = None

    for i in range(len(headers)):
        original_level, text = headers[i]

        # Check if this is a numbered header
        number_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", text)

        if number_match:
            # This is a numbered header
            number = number_match.group(1)
            parts = number.split(".")

            # Calculate level relative to the highest level
            # The highest level becomes level 2
            level_difference = len(parts) - base_level_length
            level = 2 + level_difference

            current_section_level = level
            corrected.append((level, text))
        else:
            # This is an unnumbered header

            # Check if there's a numbered header after this one
            has_following_numbered = False
            next_numbered_level = None

            for j in range(i + 1, len(headers)):
                next_text = headers[j][1]
                next_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", next_text)
                if next_match:
                    has_following_numbered = True
                    next_number = next_match.group(1)
                    next_parts = next_number.split(".")

                    # Calculate level using the same relative approach
                    level_difference = len(next_parts) - base_level_length
                    next_numbered_level = 2 + level_difference
                    break

            # Between numbered headers
            if current_section_level is not None and has_following_numbered:
                # It's between two numbered headers
                # Assign level one below the minimum of the enclosing levels
                enclosing_level = min(current_section_level, next_numbered_level)
                corrected.append((enclosing_level + 1, text))
            else:
                # Default to level 2
                corrected.append((2, text))

        # Replace "## Notes:", "## Notes", or "## Note" with "Notes:"
        if text.lower() in ["## notes:", "## notes", "## note"]:
            corrected.append((2, "Notes:"))

    return corrected


if __name__ == "__main__":
    markdown_dir = Path("data/processed/43-101")
    sample_files = list(markdown_dir.glob("*.md"))[4]  # Get first file
    with open(sample_files, "r", encoding="utf-8") as f:
        markdown_content = f.read()
    headers = extract_headers_from_markdown(markdown_content)
    corrected_headers = correct_header_levels(headers)

    print("\nCorrected headers:")
    for level, text in corrected_headers:
        print(f"{'  ' * (level - 1)}{'#' * level} {text}")


Corrected headers:
  ## NOTE
  ## REPORT NI 43-101
  ## TECHNICAL REPORT ON THE
  ## MINERAL RESOURCES AND RESERVES OF THE
  ## LOS SANTOS MINE PROJECT,
  ## SPAIN
  ## TABLE OF CONTENTS
  ## LIST OF TABLES
  ## LIST OF FIGURES
  ## APPENDICES
  ## A Geostatistical Plots
  ## B Glossary of Terms
  ## 1 SUMMARY
    ### 1.1 Introduction and Overview
    ### 1.2 Ownership
    ### 1.3 Geology and Mineralization
    ### 1.4 Database and Resource Estimation
    ### 1.5 Mine Planning
    ### 1.6 Mineral Processing
    ### 1.7 Mineral Resource and Reserve Estimates
      #### Notes
    ### 1.8 Conclusions
  ## 2 INTRODUCTION
    ### 2.1 Introduction
    ### 2.2 Terms of Reference
    ### 2.3 Sources of Information
    ### 2.4 Units and Currency
    ### 2.5 Disclaimer
  ## 3 RELIANCE ON OTHER EXPERTS
  ## 4 PROPERTY DESCRIPTION AND LOCATION
  ## 5 ACCESSIBILITY, CLIMATE, LOCAL RESOURCES, INFRASTRUCTURE, PHYSIOGRAPHY
  ## 6 HISTORY
  ## 7 GEOLOGICAL SETTING AND MINERALIZATION
  ## 8 DEPOSIT TYP

In [4]:
# Write a function to correct and replace the headers in all markdown files in the data/processed/43-101 directory and write the corrected markdown to a new folder in the data/processed/43-101-refined directory
import os


def correct_headers_in_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for file_path in Path(input_dir).glob("*.md"):
        with open(file_path, "r", encoding="utf-8") as f:
            markdown_content = f.read()
        headers = extract_headers_from_markdown(markdown_content)
        corrected_headers = correct_header_levels(headers)
        for old_header, new_header in zip(headers, corrected_headers, strict=False):
            markdown_content = markdown_content.replace(
                f"{'#' * old_header[0]} {old_header[1]}",
                f"{'#' * new_header[0]} {new_header[1]}",
            )
        with open(
            os.path.join(output_dir, file_path.name),
            "w",
            encoding="utf-8",
        ) as f:
            f.write(markdown_content)


correct_headers_in_directory(
    input_dir="data/processed/43-101",
    output_dir="data/processed/43-101-refined",
)